# Práctica: Identificadores Persistentes (PIDs) y Metadatos – Parte I


En esta práctica aprenderás a identificar, resolver y analizar identificadores persistentes utilizados en ciencia abierta. Primero realizarás una parte guiada con PIDs reales y después resolverás ejercicios individuales donde compararás DOIs, localizarás relaciones entre objetos científicos y generarás/partearás un XML con identificadores reales.


## 🟩 Parte 1 – Parte guiada
Sigue estos pasos con los identificadores propuestos. Usa un navegador para abrir las URLs y anota tus observaciones en las celdas que prefieras.


### 1.1 Resolver diferentes tipos de identificadores
Para cada PID anota: a qué landing page resuelve, qué información contiene, si está pensada para personas/máquinas y qué relaciones aparecen.

* **Dataset (DOI DataCite/Zenodo):** https://doi.org/10.5281/zenodo.1148209
* **Artículo (Crossref DOI):** https://doi.org/10.1038/sdata.2016.18
* **Persona (ORCID):** https://orcid.org/0000-0002-1825-0097
* **Institución (ROR):** https://ror.org/038sjwq14
* **Software (SWHID – Software Heritage):** https://archive.softwareheritage.org/swh:1:dir:8dfc7305b6beec8e6d8d4c3a7c33dcb162d432c0


### 1.2 Explorar relaciones en el PID Graph (DataCite Commons)
Entra en https://commons.datacite.org/ y busca el DOI del dataset anterior. Revisa qué artículos lo citan, qué software o datasets están relacionados, quiénes son sus creadores (ORCID) y qué instituciones (ROR) aparecen asociadas.


### 1.3 Consulta básica a la API de DataCite
Abre https://api.datacite.org/dois/10.5281/zenodo.1148209 y localiza el título, creadores, licencia, identificadores relacionados y fecha de publicación.
Puedes usar la celda siguiente para hacer la misma consulta desde Python con `requests` y explorar el JSON.


In [ ]:
import requests

doi = "10.5281/zenodo.1148209"
endpoint = f"https://api.datacite.org/dois/{doi}"
response = requests.get(endpoint, headers={"Accept": "application/vnd.api+json"})
print(f'Código de estado: {response.status_code}')
data = response.json() if response.ok else {}
attributes = data.get('data', {}).get('attributes', {})
print('Título:', attributes.get('titles', [{}])[0].get('title'))
print('Creadores:', [creator.get('name') for creator in attributes.get('creators', [])])
print('Licencia:', attributes.get('license', attributes.get('rightsList', [{}])[0].get('rightsIdentifier')))
print('Identificadores relacionados:', attributes.get('relatedIdentifiers'))
print('Fecha de publicación:', attributes.get('published') or attributes.get('publicationYear'))


## 🟧 Parte 2 – Ejercicios individuales
Trabaja de forma autónoma a partir de aquí. Usa las celdas en blanco o añade nuevas para tus notas y resultados.


### 🧪 Ejercicio 1 – Comparar dos DOIs
1. Elige un DOI de Zenodo bien documentado.
2. Elige un DOI de otro repositorio con metadatos más simples.
3. Evalúalos con la tabla mental (metadatos completos, licencia visible, ORCID, relaciones, descripción, enlaces a archivos).
4. Comenta cuál es mejor, qué le falta al DOI más débil y qué impacto tiene en FAIR y reutilización.

Puedes usar esta tabla como guía y completarla en una celda aparte:

| DOI | Metadatos completos | Licencia visible | ORCID | Relaciones | Descripción | Enlaces claros |
| --- | --- | --- | --- | --- | --- | --- |


### 🧪 Ejercicio 2 – Artículo con dataset y software vinculado
Busca un artículo cuyo DOI incluya un dataset asociado y un software vinculado. Anota el PID del artículo, del dataset, del software y cómo están relacionados (citación, `isSupplementedBy`, `isReferencedBy`, etc.).


### 🧪 Ejercicio 3 – Generar un documento XML (sin namespace)
Completa el diccionario de metadatos con identificadores reales y ejecuta la celda para generar un XML (`dataset_pid_story.xml`).
Incluye un dataset (DOI), autores con ORCID, institución con ROR, software con SWHID, proyecto (RAiD/DOI), licencia, fechas, versionado y enlaces relevantes.


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

metadata = {
    'dataset_title': 'Zenodo record 1148209 (sustituye por el título real)',
    'dataset_pid': '10.5281/zenodo.1148209',
    'creators': [
        {'name': 'Tim Berners-Lee', 'orcid': '0000-0002-1825-0097'}
    ],
    'institution': {'name': 'CERN', 'ror': 'https://ror.org/01ggx4157'},
    'software': {'name': 'Ejemplo Software Heritage', 'swhid': 'swh:1:dir:8dfc7305b6beec8e6d8d4c3a7c33dcb162d432c0'},
    'project': {'title': 'Horizon 2020 (actualiza con tu proyecto)', 'pid': '10.13039/100010661'},
    'license': 'CC-BY-4.0',
    'publication_date': '2018-01-01',
    'version': '1.0',
    'related_links': ['https://doi.org/10.1038/sdata.2016.18']
}

dataset = ET.Element('dataset')
ET.SubElement(dataset, 'title').text = metadata['dataset_title']
ET.SubElement(dataset, 'pid').text = metadata['dataset_pid']
ET.SubElement(dataset, 'version').text = metadata['version']
ET.SubElement(dataset, 'license').text = metadata['license']
ET.SubElement(dataset, 'publicationDate').text = metadata['publication_date']

for creator in metadata['creators']:
    creator_el = ET.SubElement(dataset, 'creator')
    ET.SubElement(creator_el, 'name').text = creator['name']
    ET.SubElement(creator_el, 'orcid').text = creator['orcid']

institution = ET.SubElement(dataset, 'institution')
ET.SubElement(institution, 'name').text = metadata['institution']['name']
ET.SubElement(institution, 'ror').text = metadata['institution']['ror']

software = ET.SubElement(dataset, 'software')
ET.SubElement(software, 'name').text = metadata['software']['name']
ET.SubElement(software, 'swhid').text = metadata['software']['swhid']

project = ET.SubElement(dataset, 'project')
ET.SubElement(project, 'title').text = metadata['project']['title']
ET.SubElement(project, 'pid').text = metadata['project']['pid']

links = ET.SubElement(dataset, 'links')
for link in metadata['related_links']:
    ET.SubElement(links, 'relatedIdentifier').text = link

def indent(elem, level=0):
    spacer = '    '
    i = '
' + level * spacer
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + spacer
        for child in elem:
            indent(child, level + 1)
        if not child.tail or not child.tail.strip():
            child.tail = i
    if level and (not elem.tail or not elem.tail.strip()):
        elem.tail = i

indent(dataset)
tree = ET.ElementTree(dataset)
output_path = Path('dataset_pid_story.xml')
tree.write(output_path, encoding='utf-8', xml_declaration=True)
print(f'Se creó el XML en {output_path.resolve()}')


### 🧪 Ejercicio 4 – Parsear tu XML y contar la historia del dataset
Ejecuta la celda siguiente para leer el XML generado y producir un texto narrativo con los identificadores. Puedes adaptar el formato del relato según necesites.


In [ ]:
tree = ET.parse('dataset_pid_story.xml')
root = tree.getroot()

title = root.findtext('title')
pid = root.findtext('pid')
creators = [f"{c.findtext('name')} (ORCID {c.findtext('orcid')})" for c in root.findall('creator')]
institution = root.find('institution')
inst_text = f"{institution.findtext('name')} (ROR {institution.findtext('ror')})" if institution is not None else ''
software = root.find('software')
software_text = f"{software.findtext('name')} (SWHID {software.findtext('swhid')})" if software is not None else ''
project = root.find('project')
project_text = f"{project.findtext('title')} ({project.findtext('pid')})" if project is not None else ''
license_text = root.findtext('license')
pub_date = root.findtext('publicationDate')
version = root.findtext('version')
related = [link.text for link in root.findall('links/relatedIdentifier')]

story = (
    f"El dataset '{title}' con ID {pid} (versión {version}) fue creado por "
    f"{', '.join(creators)}. Está afiliado a {inst_text}. "
    f"Se asocia al software {software_text} y forma parte del proyecto {project_text}. "
    f"La licencia aplicada es {license_text} y se publicó el {pub_date}. "
    f"Enlaces relacionados: {', '.join(related)}."
)

print(story)
